# 00 — Polar AccessLink: Fetch RR Intervals

Este notebook usa a API Polar AccessLink v3 para descarregar os intervalos RR (batimento-a-batimento em ms) de cada participante.

**Fluxo por utilizador:**
1. Registo do utilizador na API (idempotente — 409 = já registado, OK)
2. Abertura de uma *exercise transaction* para aceder aos treinos novos
3. Listagem dos exercícios disponíveis na transaction
4. Download dos RR intervals por exercício
5. Commit obrigatório da transaction
6. Guarda cada sessão em `data/raw/<participante>/rr/<user_id>_<date>_RR.csv`
7. Cálculo de métricas HRV: RMSSD, lnRMSSD, HR médio de repouso

> **Nota:** A transaction consome os exercícios — na próxima execução só aparecerão sessões **novas** desde o último commit.

In [1]:
import sys, logging, math
from pathlib import Path

import pandas as pd
import yaml

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.polar_accesslink import get_rr_intervals

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

TOKENS_PATH = REPO_ROOT / 'config' / 'tokens.yaml'
RAW_DIR     = REPO_ROOT / 'data' / 'raw'

with open(TOKENS_PATH, encoding='utf-8') as fh:
    tokens_cfg = yaml.safe_load(fh)

participants = tokens_cfg['participants']
print(f'Participantes carregados: {list(participants.keys())}')

Participantes carregados: ['polar01', 'polar02', 'polar03']


In [2]:
# Fetch RR intervals for every participant
all_sessions: list[dict] = []

for pid, cfg in participants.items():
    print(f'\n── {pid} (user_id={cfg["user_id"]}) ──')
    try:
        sessions = get_rr_intervals(
            access_token=cfg['access_token'],
            user_id=cfg['user_id'],
        )
        for s in sessions:
            s['participant'] = pid
            all_sessions.append(s)
        if sessions:
            for s in sessions:
                print(f'  exercise {s["exercise_id"]} | {s["date"]} | {len(s["rr_intervals"])} RR intervals')
        else:
            print('  → sem sessões novas')
    except Exception as exc:
        print(f'  ERRO: {exc}')

print(f'\nTotal de sessões com RR: {len(all_sessions)}')


── polar01 (user_id=64476620) ──


INFO | No new exercises for user 64476620.


  → sem sessões novas

── polar02 (user_id=64480968) ──


INFO | Transaction 333734944 opened for user 64480968.


INFO | Transaction 333734944 committed → HTTP 200.


  ERRO: 404 Client Error:  for url: https://www.polaraccesslink.com/v3/users/64480968/exercise-transactions/333734944/exercises

── polar03 (user_id=64480991) ──


INFO | No new exercises for user 64480991.


  → sem sessões novas

Total de sessões com RR: 0


In [3]:
# Save each session's RR intervals to data/raw/<participant>/rr/<user_id>_<date>_RR.csv
saved = []

for session in all_sessions:
    if not session['rr_intervals']:
        continue

    pid      = session['participant']
    user_id  = session['user_id']
    date     = session['date'] or 'unknown'
    rr       = session['rr_intervals']

    out_dir  = RAW_DIR / pid / 'rr'
    out_dir.mkdir(parents=True, exist_ok=True)

    filename = f"{user_id}_{date}_RR.csv"
    out_path = out_dir / filename

    pd.DataFrame({'rr_ms': rr}).to_csv(out_path, index=False)
    saved.append(out_path)
    print(f'Guardado → {out_path.relative_to(REPO_ROOT)}  ({len(rr)} linhas)')

if not saved:
    print('Nenhum ficheiro guardado — sem sessões com RR intervals.')

Nenhum ficheiro guardado — sem sessões com RR intervals.


In [4]:
# HRV metrics per session
if not all_sessions or not any(s['rr_intervals'] for s in all_sessions):
    print('Sem dados RR — execute novamente após sincronizar sessões no Polar Flow.')
else:
    rows = []
    for s in all_sessions:
        rr = s['rr_intervals']
        if len(rr) < 2:
            continue

        diffs    = [abs(rr[i+1] - rr[i]) for i in range(len(rr)-1)]
        rmssd    = math.sqrt(sum(d**2 for d in diffs) / len(diffs))
        ln_rmssd = math.log(rmssd) if rmssd > 0 else float('nan')
        hr_mean  = 60000 / (sum(rr) / len(rr))

        rows.append({
            'participant':  s['participant'],
            'exercise_id':  s['exercise_id'],
            'date':         s['date'],
            'n_rr':         len(rr),
            'rr_mean_ms':   round(sum(rr)/len(rr), 1),
            'HR_rest_bpm':  round(hr_mean, 1),
            'RMSSD_ms':     round(rmssd, 2),
            'lnRMSSD':      round(ln_rmssd, 3),
        })

    if rows:
        df_hrv = pd.DataFrame(rows)
        display(df_hrv)
    else:
        print('Sessões encontradas mas sem RR intervals suficientes para calcular HRV.')

Sem dados RR — execute novamente após sincronizar sessões no Polar Flow.
